In [22]:
import numpy as np
np.random.seed(42)

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

In [23]:
from indices import *

import tracemalloc
import time

In [24]:
data = [None]*2
data[0] = np.load("dataset_cropped_64_npy\\TRAIN_HEALTHY_even.npy") # healthy
data[1] = np.load("dataset_cropped_64_npy\\TRAIN_STRESSED_even.npy") # stressed

In [25]:
labels = np.concatenate([np.zeros(data[0].shape[1]), np.ones(data[1].shape[1])])
labels.shape

(38978,)

In [26]:
data[1].shape

(12, 19489)

In [27]:
tracemalloc.start()

In [28]:
band_indexes = list(range(1, 12))
encoder = IndicesClassEncoderEq([NORMP], band_indexes)

feature_id = []
features = []
for i in range(encoder.total_length):
    index = encoder.getIndex(i)
    a = index.args
    if a[0] < a[1]:
        continue

    feature_id.append(i)
    features.append(np.concatenate([index.getValue(data[0]), index.getValue(data[1])]))

features = np.array(features).swapaxes(0, 1)

In [29]:
features.shape

(38978, 66)

In [30]:
time_start = time.time()

dt = RandomForestClassifier(random_state=42)
dt.fit(features, labels)

result = permutation_importance(
    dt, features, labels, random_state=42, n_jobs=8
)



In [31]:
selected = np.array(result.importances_mean).argsort()[::-1][:3]

In [32]:
time_end = time.time()
print("Time:", time_end - time_start, "sec")
print("MEM usage:", np.array(tracemalloc.get_traced_memory()) / 1024**2, "mb")
tracemalloc.stop()

Time: 94.12406015396118 sec
MEM usage: [19.8338995 99.5966053] mb


In [33]:
mapping = {
    0: "B1",
    1: "B2",
    2: "B3",
    3: "B4",
    4: "B5",
    5: "B6",
    6: "B7",
    7: "B8",
    8: "B8A",
    9: "B9",
    10: "B11",
    11: "B12"
}

for id in selected:
    index_id = feature_id[id]
    index = encoder.getIndex(index_id)
    name = getIndexName(index, mapping)
    print("Id:", index_id, "Name:", name)

Id: 13 Name: NORMP(B4, B3)
Id: 53 Name: NORMP(B11, B6)
Id: 54 Name: NORMP(B12, B6)
